In [7]:
import os
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
import requests
from io import BytesIO
import pandas as pd

In [ ]:
# Load the BLIP model and processor from Hugging Face, do this once at the start
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

In [ ]:
def generate_image_captions(image_folder, sample_size=50):
    """
    Takes a folder of images, generates captions for a random sample, and returns a list of captions.
    """
    captions = []
    image_files = os.listdir(image_folder)[:sample_size]  # Limit to sample size
    
    for file in image_files:
        raw_image = Image.open(os.path.join(image_folder, file)).convert("RGB")
        
        inputs = processor(raw_image, return_tensors="pt")
        out = model.generate(**inputs)
        caption = processor.decode(out[0], skip_special_tokens=True)
        
        captions.append(caption)
        
    return captions

In [5]:
test = generate_image_captions("./e-commerce/Apparel/Boys/Images/images_with_product_ids")
print(test)

["a boy ' s blue striped t - shirt with a sail and a sail", 't - shirt with print', 't - shirt with print', 't - shirt with cat print', 't - shirt be happy', 'a black t - shirt with a green star on the front', 't - shirt with a cartoon character', 'a picture of a grey cargo shorts', 'a picture of a denim shorts with a skull on the side', 'a red and white shirt with a small patch on the chest', 'a yellow shorts with a black logo on the side', 'a woman wearing a belted belt and shorts', "girls ' white t - shirt with yellow and black print", 'jeans with pockets and pockets', 't - shirt with print', "a boy ' s shirt with a blue and white checkered pattern", 'a dog wearing a hat and a vest', 't - shirt with print', 'marvel t - shirt for boys', "a red and white striped shirt with the word ' happy ' on it", 't - shirt with print', 't - shirt with print and logo', 'a red shorts with a drawset and a drawset', 'a grey vest with black trimmings and a black collar', 't - shirt with tiger print', '

In [10]:
# for handling URLs instead of local files:
def generate_image_url_captions(df, url_column, sample_size=50):
    """
    Takes a pandas DataFrame and a column of image URLs, fetches a random sample,
    and returns a list of generated captions.
    """
    captions = []
    valid_df = df.dropna(subset=[url_column])
    sample_df = valid_df.sample(n=min(sample_size, len(valid_df)))
    
    for index, row in sample_df.iterrows():
        url = row[url_column]
        
        try:
            response = requests.get(url, timeout=5)
            response.raise_for_status()
            
            raw_image = Image.open(BytesIO(response.content)).convert('RGB')
            
            inputs = processor(raw_image, return_tensors="pt")
            out = model.generate(**inputs)
            caption = processor.decode(out[0], skip_special_tokens=True)
            
            captions.append(caption)
        except Exception as e:
            print(f"Skipped image at index {index} due to error: {e}")
            continue
        
    return captions

In [11]:
# example usage:
amazon_df = pd.read_csv("./amazon.csv")
captions = generate_image_url_captions(df=amazon_df, url_column="img_link", sample_size=10)

Skipped image at index 317 due to error: 400 Client Error: Bad Request for url: https://m.media-amazon.com/images/W/WEBP_402378-T1/images/I/41sA8PA31pL._SY300_SX300_QL70_FMwebp_.jpg
Skipped image at index 1235 due to error: 400 Client Error: Bad Request for url: https://m.media-amazon.com/images/W/WEBP_402378-T2/images/I/41svI04SS1L._SX300_SY300_QL70_FMwebp_.jpg
Skipped image at index 244 due to error: 400 Client Error: Bad Request for url: https://m.media-amazon.com/images/W/WEBP_402378-T2/images/I/41rEpW57SyL._SX300_SY300_QL70_FMwebp_.jpg
Skipped image at index 694 due to error: 400 Client Error: Bad Request for url: https://m.media-amazon.com/images/W/WEBP_402378-T1/images/I/41XH-IpxCQL._SX300_SY300_QL70_FMwebp_.jpg


In [12]:
print(captions)

['the adjustable phone stand with adjustable base', 'the black steel stand is shown with two legs', 'the apple watch is shown in a blue strap', 'usb cable usb type c type c type c type c type c type c type c type c type', 'a blender with a blender and a blender', 'the water dispe is a water dispe that is used to clean and dispe']
